# U-Net Building Segmentation Baseline

This notebook trains a **binary semantic-segmentation** model on the 1024×1024 image/mask tiles produced by `01_satellite_image_exploration.ipynb`.

**Input:** RGB aerial-image tile  
**Target:** one binary mask (`0 = background`, `1 = building`)  
**Model:** U-Net with an ImageNet-pretrained ResNet34 encoder

The reported baseline uses 754 tiles from one `33cae6` geographic scene. The 80/20 validation split is therefore a **same-scene held-out tile evaluation**, not a cross-location generalization test.

## 1. Environment and dataset paths

The training run was performed in a Kaggle Notebook using an NVIDIA Tesla T4 GPU. The Kaggle input dataset contains:

```text
data/images/   # 754 image GeoTIFF tiles
data/masks/    # 754 binary mask GeoTIFF tiles
```

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

KAGGLE_INPUT = Path("/kaggle/input")
image_dirs = list(KAGGLE_INPUT.rglob("data/images"))
mask_dirs = list(KAGGLE_INPUT.rglob("data/masks"))

assert len(image_dirs) == 1, image_dirs
assert len(mask_dirs) == 1, mask_dirs

IMAGE_DIR = image_dirs[0]
MASK_DIR = mask_dirs[0]

image_paths = sorted(IMAGE_DIR.glob("*.tif"))
mask_paths = sorted(MASK_DIR.glob("*.tif"))

assert len(image_paths) == len(mask_paths) == 754
assert [p.name for p in image_paths] == [p.name for p in mask_paths]

print("Image tiles:", len(image_paths))
print("Mask tiles :", len(mask_paths))

## 2. Inspect one non-empty sample

Rasterio returns an image as `[bands, height, width]`. We use the first three bands as RGB. The fourth band inspected during development behaved as a constant alpha/validity band for the sampled imagery, so the baseline uses three input channels.

In [ ]:
sample_mask_path = None
for mask_path in mask_paths:
    with rasterio.open(mask_path) as src:
        mask = src.read(1)
    if mask.sum() > 0:
        sample_mask_path = mask_path
        break

sample_image_path = IMAGE_DIR / sample_mask_path.name

with rasterio.open(sample_image_path) as src:
    image = src.read()

with rasterio.open(sample_mask_path) as src:
    mask = src.read(1)

print("Tile:", sample_image_path.name)
print("Image shape:", image.shape, image.dtype)
print("Mask shape :", mask.shape, mask.dtype)
print("Mask values:", np.unique(mask))

rgb = np.transpose(image[:3], (1, 2, 0))

plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
plt.imshow(rgb)
plt.title("Satellite image (RGB)")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(mask, cmap="gray")
plt.title("Ground-truth building mask")
plt.axis("off")
plt.show()

## 3. Create a reproducible train/validation split

Many tiles contain no building pixels, so the split is stratified using `has_building`. This preserves a similar proportion of empty/non-empty tiles in training and validation.

- Total tiles: **754**
- Training: **603**
- Validation: **151**
- Random seed: **42**

Because both sets come from the same original scene, nearby tiles may still be spatially correlated. This is explicitly treated as a baseline limitation.

In [ ]:
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

records = []
for image_path, mask_path in tqdm(zip(image_paths, mask_paths), total=len(image_paths)):
    assert image_path.name == mask_path.name

    with rasterio.open(mask_path) as src:
        mask = src.read(1)

    building_fraction = (mask > 0).mean()

    records.append({
        "filename": image_path.name,
        "building_fraction": building_fraction,
        "has_building": int(building_fraction > 0),
    })

df = pd.DataFrame(records)

train_df, val_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["has_building"],
)

train_df = train_df.copy()
val_df = val_df.copy()
train_df["split"] = "train"
val_df["split"] = "val"

manifest = pd.concat([train_df, val_df], ignore_index=True)
manifest.to_csv("/kaggle/working/dataset_manifest.csv", index=False)

print(manifest["split"].value_counts())
print()
print("Building-containing tiles:")
print(manifest.groupby("split")["has_building"].agg(["sum", "count"]))
print()
print("Mean building-pixel percentage:")
print(manifest.groupby("split")["building_fraction"].mean() * 100)

## 4. PyTorch Dataset

A PyTorch `Dataset` defines how one filename becomes a model-ready sample.

For each tile:

1. read RGB bands,
2. convert `uint8` values from 0–255 to floating point,
3. apply ImageNet normalization for the pretrained ResNet34 encoder,
4. read the binary mask,
5. return tensors shaped `[3, H, W]` and `[1, H, W]`.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class BuildingSegmentationDataset(Dataset):
    def __init__(self, dataframe, image_dir, mask_dir):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.mask_dir = mask_dir

        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        filename = self.dataframe.iloc[idx]["filename"]
        image_path = self.image_dir / filename
        mask_path = self.mask_dir / filename

        with rasterio.open(image_path) as src:
            image = src.read([1, 2, 3]).astype(np.float32)

        image = image / 255.0
        image = (image - self.mean) / self.std

        with rasterio.open(mask_path) as src:
            mask = src.read(1)

        mask = (mask > 0).astype(np.float32)
        mask = np.expand_dims(mask, axis=0)

        return torch.from_numpy(image), torch.from_numpy(mask), filename

train_dataset = BuildingSegmentationDataset(train_df, IMAGE_DIR, MASK_DIR)
val_dataset = BuildingSegmentationDataset(val_df, IMAGE_DIR, MASK_DIR)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

## 5. DataLoaders

A `DataLoader` groups individual samples into batches. The baseline uses batch size 2 because each image is 1024×1024 and therefore relatively memory intensive.

In [ ]:
BATCH_SIZE = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

images, masks, filenames = next(iter(train_loader))
print("Images batch shape:", images.shape)
print("Masks batch shape :", masks.shape)

## 6. U-Net model

U-Net contains an **encoder** that progressively extracts higher-level image features and a **decoder** that upsamples those features back to pixel resolution. **Skip connections** pass fine spatial detail from encoder stages directly to matching decoder stages, helping recover building boundaries.

The network returns one raw value (**logit**) per pixel. We do not apply sigmoid inside the model because `BCEWithLogitsLoss` handles logits in a numerically stable way.

In [ ]:
# Kaggle may require Internet to be enabled for this installation and for the
# first download of pretrained ResNet34 weights.
!pip install -q segmentation-models-pytorch

In [ ]:
import segmentation_models_pytorch as smp

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None,
).to(DEVICE)

print("Device:", DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Sanity-check one forward pass.
images, masks, _ = next(iter(train_loader))
with torch.no_grad():
    outputs = model(images.to(DEVICE))

print("Input :", images.shape)
print("Target:", masks.shape)
print("Output:", outputs.shape)

## 7. Loss and optimizer

The baseline combines:

- **Binary Cross-Entropy (BCE):** pixel-by-pixel building/background classification.
- **Dice loss:** directly rewards overlap between predicted and true building regions.

Dice is useful here because only about 9% of pixels are buildings, so background strongly dominates the dataset.

In [ ]:
import torch.nn as nn

bce_loss = nn.BCEWithLogitsLoss()
dice_loss = smp.losses.DiceLoss(mode="binary", from_logits=True)

def criterion(logits, targets):
    return bce_loss(logits, targets) + dice_loss(logits, targets)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

## 8. Training and validation

During training the model parameters are updated from the 603 training tiles.

During validation:

1. the model is put in evaluation mode,
2. no gradients are computed,
3. logits are converted to probabilities using sigmoid,
4. probability ≥ 0.5 becomes building,
5. IoU, Dice, precision, and recall are accumulated globally over validation pixels.

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0

    for images, masks, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(images)
            loss = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader):
    model.eval()

    total_loss = 0.0
    intersection = 0.0
    pred_pixels = 0.0
    true_pixels = 0.0
    union = 0.0

    for images, masks, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(images)
            loss = criterion(logits, masks)

        total_loss += loss.item()

        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= 0.5).float()

        batch_intersection = (predictions * masks).sum().item()
        batch_pred = predictions.sum().item()
        batch_true = masks.sum().item()
        batch_union = batch_pred + batch_true - batch_intersection

        intersection += batch_intersection
        pred_pixels += batch_pred
        true_pixels += batch_true
        union += batch_union

    eps = 1e-7

    return {
        "loss": total_loss / len(loader),
        "iou": intersection / (union + eps),
        "dice": 2 * intersection / (pred_pixels + true_pixels + eps),
        "precision": intersection / (pred_pixels + eps),
        "recall": intersection / (true_pixels + eps),
    }

In [ ]:
EPOCHS = 5
BEST_MODEL_PATH = "/kaggle/working/best_unet_resnet34.pth"

best_dice = -1.0
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader)
    metrics = validate(model, val_loader)

    history.append({"epoch": epoch, "train_loss": train_loss, **metrics})

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {metrics['loss']:.4f} | "
        f"IoU: {metrics['iou']:.4f} | "
        f"Dice: {metrics['dice']:.4f} | "
        f"Precision: {metrics['precision']:.4f} | "
        f"Recall: {metrics['recall']:.4f}"
    )

    if metrics["dice"] > best_dice:
        best_dice = metrics["dice"]
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print("Saved new best model.")

history_df = pd.DataFrame(history)
history_df.to_csv("/kaggle/working/training_history.csv", index=False)

## 9. Final validation metrics

The reported run selected the epoch-4 checkpoint using validation Dice. The saved run produced approximately:

- IoU: **0.7954**
- Dice: **0.8860**
- Precision: **0.8956**
- Recall: **0.8767**

These are validation metrics, not training metrics.

In [ ]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
model.eval()

final_metrics = validate(model, val_loader)
print(final_metrics)

pd.DataFrame([final_metrics]).to_csv(
    "/kaggle/working/final_metrics.csv",
    index=False,
)

## 10. Training curve

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train Loss")
plt.plot(history_df["epoch"], history_df["loss"], marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid()
plt.savefig("/kaggle/working/training_loss.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Visualize a representative validation prediction

For presentation, select a validation tile with meaningful building coverage rather than only showing an unusually easy or empty example.

In [ ]:
good_samples = val_df[
    (val_df["building_fraction"] > 0.10)
    & (val_df["building_fraction"] < 0.40)
].sort_values("building_fraction", ascending=False)

filename = good_samples.iloc[0]["filename"]
sample_idx = val_dataset.dataframe.index[
    val_dataset.dataframe["filename"] == filename
].tolist()[0]

image, gt_mask, filename = val_dataset[sample_idx]

with torch.no_grad():
    logits = model(image.unsqueeze(0).to(DEVICE))
    probability = torch.sigmoid(logits)
    prediction = (probability >= 0.5).float()[0, 0].cpu().numpy()

gt_mask = gt_mask[0].numpy()

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

display_image = image.cpu() * std + mean
display_image = display_image.clamp(0, 1).permute(1, 2, 0).numpy()

plt.figure(figsize=(16, 5))

plt.subplot(1, 3, 1)
plt.imshow(display_image)
plt.title(f"Satellite Image\n{filename}")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(gt_mask, cmap="gray")
plt.title("Ground Truth")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(prediction, cmap="gray")
plt.title("U-Net Prediction")
plt.axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/segmentation_result.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Interpretation and limitation

This baseline demonstrates that the U-Net learned meaningful building segmentation on held-out `33cae6` tiles. It does **not** yet establish geographic generalization because all tiles originate from one large scene and the split was random at tile level.

The next experiment should therefore keep this trained model frozen and evaluate it on a second labeled Open Cities scene from another location before deciding whether more training data or model changes are necessary.